# Assignment 3 · Parte 1 – Visualización con Plotly

**Grupo 11** · Diplomado PUCP · Fundamentos de Python

Usamos **Plotly Express**: gráficos interactivos (al pasar el mouse aparece el dato) que se pueden exportar a HTML.

## 1. Cargar los datos

- El archivo está en la carpeta `assigment_3/datos/`, como pide la estructura del trabajo.
- Leemos el dataset.csv del Assigment_2

In [1]:
# Cargamos librerías: pandas para manejar la tabla y plotly.express (px) para graficar
import pandas as pd
import plotly.express as px

# Leemos el dataset del Assignment 2.
# dtype={"ubigeo": str} evita que '01' se convierta en el número 1.
dataset = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})

# Texto de la fuente: lo pegamos al pie de cada gráfico (la rúbrica pide citar la fuente)
#FUENTE = "Fuente: dataset.csv del Assignment 2, grupo 11 (declaratorias de emergencia y lluvia en la capital de cada departamento, 1 ene – 31 may 2024)."

- Mostramos las dimensiones y las primeras 5 filas.

In [2]:
# .shape devuelve (filas, columnas): 25 departamentos y 9 variables
dataset.shape

(25, 9)

In [3]:
# Primeras 5 filas para revisar que la tabla se leyó bien
dataset.head(5)

,ubigeo,departamento,declaratorias,prorrogas,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,01,Amazonas,4,2,Chachapoyas,-6.23169,-77.86903,455.0,0
1,02,Áncash,5,5,Huaraz,-9.52614,-77.52869,1058.3,3
2,03,Apurímac,7,4,Abancay,-13.63426,-72.88380,670.6,2
3,04,Arequipa,7,6,Arequipa,-16.39899,-71.53747,235.3,1
4,05,Ayacucho,7,5,Ayacucho,-13.16380,-74.22345,467.5,1


## 2. Pregunta y columnas

**Temporada analizada:** 1 de enero al 31 de mayo de 2024 (temporada asignada al grupo 11 en el Assignment 2).

**Pregunta:** ¿Los departamentos donde más llovió son también los que recibieron más declaratorias de emergencia?

| Columna | Qué mide |
|---|---|
| `departamento` | nombre del departamento (25) |
| `declaratorias` | número de declaratorias de estado de emergencia |
| `prorrogas` | número de prórrogas de esas declaratorias |
| `lluvia_total_mm` | lluvia acumulada del 1 de enero al 31 de mayo de 2024, medida en la capital del departamento |
| `dias_lluvia_fuerte` | número de días, entre el 1 de enero y el 31 de mayo de 2024 (152 días), con lluvia fuerte en la capital |

## 3. Gráficos interactivos con Plotly Express

### Gráfico 1 · Barras: top 10 de declaratorias por departamento

Primero ordenamos la tabla con `sort_values()` para ver el ranking.

In [4]:
# Ordenamos de mayor a menor (ascending=False) y nos quedamos con los 10 primeros (head(10))
dep_declaratorias = dataset.sort_values("declaratorias", ascending=False).head(10)

# Mostramos solo las columnas que nos interesan para este gráfico
dep_declaratorias[["departamento", "declaratorias", "prorrogas", "lluvia_total_mm"]]

,departamento,declaratorias,prorrogas,lluvia_total_mm
2,Apurímac,7,4,670.6
7,Cusco,7,5,601.0
3,Arequipa,7,6,235.3
4,Ayacucho,7,5,467.5
8,Huancavelica,7,4,819.2
14,Lima,6,5,12.2
9,Huánuco,6,3,416.7
1,Áncash,5,5,1058.3
11,Junín,5,1,1276.9
20,Puno,5,2,728.6


In [9]:
fig = px.bar(
    dep_declaratorias,
    x="departamento", y="declaratorias",       # departamento en el eje X, número de declaratorias en el eje Y
    text_auto=True,                            # escribe el valor encima de cada barra
    color="declaratorias",                     # el color depende del número de declaratorias...
    color_continuous_scale="reds",             # ...con una escala de rojos (más oscuro = más declaratorias)
    # <br> parte el título en dos líneas para que entre completo en el cuadro
    title="La sierra sur concentra el máximo de declaratorias de emergencia (7)",
    labels={"departamento": "Departamento",    # nombres claros para los ejes
            "declaratorias": "Declaratorias de emergencia"},
)
fig.update_traces(textposition="outside", cliponaxis=False)   # el número va fuera de la barra y no se corta
fig.update_layout(coloraxis_showscale=False,                  # ocultamos la barra de color: ya se ve el número
                  template="plotly_white",                    # fondo blanco, más limpio
                  title_font_size=16,
                  margin=dict(b=100))                         # espacio abajo para la fuente
# Texto de la fuente: lo pegamos al pie de cada gráfico (la rúbrica pide citar la fuente)
FUENTE = "Fuente: dataset.csv del Assignment 2, grupo 11 (declaratorias de emergencia y lluvia en la capital de cada departamento, 1 ene – 31 may 2024)."
# Agregamos la fuente al pie del gráfico (xref/yref="paper" = posición relativa al cuadro)
fig.add_annotation(text=FUENTE, xref="paper", yref="paper", x=0, y=-0.32,
                   showarrow=False, font=dict(size=10, color="gray"), xanchor="left")
fig.show()

### Gráfico 1b · Barras horizontales: los 10 departamentos con más lluvia

Barras **horizontales** (`orientation="h"`), como en la clase: los nombres se leen sin girar la cabeza.
El color indica cuántas declaratorias tuvo cada uno, así vemos si "más lluvia" coincide con "más declaratorias".

In [10]:
# nlargest(10, ...) toma los 10 departamentos con MÁS lluvia.
# Luego los ordenamos de menor a mayor para que, en barras horizontales, el más lluvioso quede arriba.
dep_lluvia = dataset.nlargest(10, "lluvia_total_mm").sort_values("lluvia_total_mm")

fig = px.bar(
    dep_lluvia,
    x="lluvia_total_mm", y="departamento",
    orientation="h",                           # barras horizontales
    text_auto=",.0f",                          # valor con separador de miles y sin decimales (1,362)
    color="declaratorias", color_continuous_scale="reds",
    title="De los 10 departamentos con más lluvia, solo Huancavelica llegó al máximo de 7 declaratorias",
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)",
            "departamento": "",                # el nombre del eje sobra: ya se leen los departamentos
            "declaratorias": "Declaratorias"},
)
fig.update_traces(textposition="outside", cliponaxis=False)
# Ampliamos el eje X un 15 % para que el número de la barra más larga no se salga del cuadro
fig.update_xaxes(range=[0, dep_lluvia["lluvia_total_mm"].max() * 1.15])
fig.update_layout(template="plotly_white", title_font_size=16, margin=dict(b=90))
fig.add_annotation(text=FUENTE, xref="paper", yref="paper", x=0, y=-0.22,
                   showarrow=False, font=dict(size=10, color="gray"), xanchor="left")
fig.show()